## TP 4 — COVID-19 Detection in Chest X-Ray Images

### Objetivo
Desarrollar un sistema de **clasificación automática de imágenes médicas** capaz de detectar casos de **COVID-19** a partir de **radiografías de tórax (Chest X-Ray)**, utilizando técnicas de **procesamiento de imágenes** y **aprendizaje automático clásico**.

### Descripción del Dataset
Se dispone de un conjunto de datos compuesto por **más de 600 imágenes** de radiografías de tórax, organizadas en tres categorías:

- **COVID-19**
- **Normal**
- **Neumonía**

Las imágenes se encuentran estructuradas en carpetas, una por cada clase.

Repositorio del proyecto:  
https://github.com/manlio99/Materia-de-aprendizaje/blob/master/3_MidtermProjects/ProjectCXR

### Consigna
1. **Preprocesamiento de las imágenes**
   - Lectura de las imágenes.
   - Normalización y/o redimensionamiento si fuera necesario.
   - Conversión a escala de grises

2. **Extracción de características**
   - Diseñar e implementar un **extractor de características** adecuado para imágenes médicas.
   - Se pueden emplear descriptores clásicos como:
     - Histogramas de intensidad
     - HOG (Histogram of Oriented Gradients)
     - Textura (LBP, Haralick, etc.)
   - Justificar brevemente la elección del método.

3. **Entrenamiento del clasificador**
   - Utilizar las características extraídas para entrenar un **clasificador supervisado**.
   - Algunos modelos sugeridos:
     - K-Nearest Neighbors (KNN)
     - Support Vector Machines (SVM)
     - Random Forest
     - Naive Bayes

4. **Evaluación del modelo**
   - Evaluar el desempeño utilizando métricas apropiadas:
     - Accuracy
     - Precision
     - Recall
     - F1-score
   - Analizar los resultados obtenidos y discutir posibles mejoras.

In [1]:
import os
import numpy as np
import pandas as pd
import requests
import time
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from tqdm import tqdm
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from glob import glob
from pathlib import Path

from skimage import io, color
from skimage.transform import resize
from skimage.exposure import rescale_intensity
from skimage.feature import hog
from skimage.feature import local_binary_pattern

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix
)

from utils import *

warnings.filterwarnings("ignore")

In [2]:
path_data = os.path.join(os.getcwd(), "dataug")
path = Path(path_data)
path.mkdir(parents=True, exist_ok=True)

In [3]:
import zipfile

with zipfile.ZipFile(path_data + "\\dataset.zip", "r") as zip_ref:
    
    zip_ref.extractall(path_data)

In [7]:
def list_image_files(root: Path, exts=(".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
    files = []
    for ext in exts:
        files += list(root.rglob(f"*{ext}"))
        files += list(root.rglob(f"*{ext.upper()}"))
    return sorted(set(files))

def build_index_from_class_folders(data_dir: Path):
    class_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
    if not class_dirs:
        raise ValueError(f"No hay subcarpetas dentro de {data_dir} (se esperan carpetas por clase).")

    class_names = [p.name for p in class_dirs]
    idx_to_class = {i: n for i, n in enumerate(class_names)}

    paths, labels = [], []
    for i, cname in enumerate(class_names):
        imgs = list_image_files(data_dir / cname)
        if len(imgs) == 0:
            print(f"[WARN] Carpeta vacía: {cname}")
            continue
        paths.extend(imgs)
        labels.extend([i] * len(imgs))

    paths = np.array(paths, dtype=object)
    y = np.array(labels, dtype=np.int32)

    return paths, y, idx_to_class

paths, y, idx_to_class = build_index_from_class_folders(path)

print("Total imágenes:", len(paths))
print("Clases:", idx_to_class)
u, c = np.unique(y, return_counts=True)
print("Distribución:", {idx_to_class[int(k)]: int(v) for k, v in zip(u, c)})

Total imágenes: 603
Clases: {0: 'COVID_19', 1: 'Normal', 2: 'Pneumonia'}
Distribución: {'COVID_19': 221, 'Normal': 234, 'Pneumonia': 148}
